# M-κ Interactive State Visualization

Demonstrates the `MKappaStateProfiles` class that combines M-κ curves with stress-strain profile visualization.

## Features

- Interactive visualization showing M-κ curve alongside current cross-section state
- Explore different points on the M-κ curve
- See strain and stress distributions at any curvature
- Force resultants displayed with visual arrows

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Import required modules
from bmcs_cross_section.cs_design.shapes import RectangularShape
from bmcs_cross_section.cs_design.reinforcement import ReinforcementLayout, ReinforcementLayer
from bmcs_cross_section.cs_design.cross_section import CrossSection
from bmcs_cross_section.matmod.ec2_concrete import EC2Concrete
from bmcs_cross_section.matmod.steel_reinforcement import SteelReinforcement
from bmcs_cross_section.mkappa.mkappa import MKappaAnalysis, create_default_mkappa
from bmcs_cross_section.mkappa.mkappa_state_profiles import MKappaStateProfiles

print("All imports successful!")

## Example 1: Default Cross-Section

Start with a standard 300×500 mm rectangular section with C30/37 concrete and B500B reinforcement.

In [ ]:
# Create and solve M-κ analysis
mkappa = create_default_mkappa()

print(f"Cross-section: {mkappa.cs.shape.b:.0f} × {mkappa.cs.shape.h:.0f} mm")
print(f"Concrete: f_ck = {mkappa.cs.concrete.f_ck:.0f} MPa")
print(f"Reinforcement:")
for i, layer in enumerate(mkappa.cs.reinforcement.layers):
    print(f"  Layer {i+1}: z = {layer.z:.0f} mm, A_s = {layer.A_s:.1f} mm²")

print("\nSolving M-κ relationship...")
mkappa.solve()
print(f"Solved {len(mkappa.kappa_values)} points")
print(f"M_max = {mkappa.M_values.max():.2f} kN·m")

## Interactive State at Mid-Point

Visualize the cross-section state at the mid-point of the M-κ curve.

In [ ]:
# Create interactive state visualization
mkappa_state = MKappaStateProfiles(mkappa=mkappa)

print(f"Current state (mid-point):")
print(f"  κ = {mkappa_state.kappa*1000:.4f} ‰/mm")
print(f"  M = {mkappa_state._get_M_at_kappa(mkappa_state.kappa):.2f} kN·m")

# Plot combined visualization
fig, ax_strain, ax_stress, ax_mk = mkappa_state.plot_mkappa_state(figsize=(18, 6))
plt.show()

## State at Peak Moment

Examine the cross-section state when the moment reaches its maximum value.

In [ ]:
# Find peak moment
idx_max = np.argmax(mkappa.M_values)
kappa_at_peak = mkappa.kappa_values[idx_max]
M_peak = mkappa.M_values[idx_max]

print(f"Peak moment state:")
print(f"  κ = {kappa_at_peak*1000:.4f} ‰/mm")
print(f"  M = {M_peak:.2f} kN·m")

# Update state to peak
mkappa_state.set_kappa(kappa_at_peak)
fig, ax_strain, ax_stress, ax_mk = mkappa_state.plot_mkappa_state(figsize=(18, 6))
plt.show()

## Progressive States Along M-κ Curve

Visualize the evolution of strain and stress distributions at 25%, 50%, 75%, and 100% of maximum curvature.

In [ ]:
# Show states at key percentages
print("Generating visualizations at 25%, 50%, 75%, and 100% of max curvature...")
mkappa_state.plot_at_percentages(
    percentages=[0.25, 0.50, 0.75, 1.0],
    figsize=(18, 6),
    show_resultants=True
)

## Example 2: Larger Cross-Section

Compare with a 400×600 mm section using C50/60 concrete and heavier reinforcement.

In [ ]:
# Create larger section
shape2 = RectangularShape(b=400.0, h=600.0)
concrete2 = EC2Concrete(f_ck=50.0)  # C50/60
steel_mat2 = SteelReinforcement(f_sy=500.0)

# Bottom and top reinforcement
layer_bottom2 = ReinforcementLayer(z=60.0, A_s=2513.3, material=steel_mat2)  # 8Ø20
layer_top2 = ReinforcementLayer(z=540.0, A_s=804.2, material=steel_mat2)     # 4Ø16

reinforcement2 = ReinforcementLayout(layers=[layer_bottom2, layer_top2])

cs2 = CrossSection(
    shape=shape2,
    concrete=concrete2,
    reinforcement=reinforcement2
)

# Solve M-κ
mkappa2 = MKappaAnalysis(cs=cs2, n_kappa=100)
mkappa2.solve()

print(f"Section 2: {cs2.shape.b:.0f} × {cs2.h_total:.0f} mm, C{concrete2.f_ck:.0f}")
print(f"M_max = {mkappa2.M_values.max():.2f} kN·m")

## State Comparison at Peak Moment

Compare the stress-strain profiles at peak moment for both sections.

In [ ]:
# Create state visualization for second section
mkappa_state2 = MKappaStateProfiles(mkappa=mkappa2)

# Set to peak moment
idx_max2 = np.argmax(mkappa2.M_values)
kappa_at_peak2 = mkappa2.kappa_values[idx_max2]

mkappa_state2.set_kappa(kappa_at_peak2)

print(f"Section 2 at peak:")
print(f"  κ = {kappa_at_peak2*1000:.4f} ‰/mm")
print(f"  M = {mkappa2.M_values[idx_max2]:.2f} kN·m")

fig, ax_strain, ax_stress, ax_mk = mkappa_state2.plot_mkappa_state(figsize=(18, 6))
plt.show()

## Exploring Early Elastic State

Look at the cross-section behavior in the elastic range (10% of max curvature).

In [ ]:
# Set to early elastic state (10% of max curvature)
kappa_elastic = mkappa.kappa_values[-1] * 0.10

print(f"Early elastic state (10% of max κ):")
print(f"  κ = {kappa_elastic*1000:.4f} ‰/mm")

mkappa_state.set_kappa(kappa_elastic)
fig, ax_strain, ax_stress, ax_mk = mkappa_state.plot_mkappa_state(figsize=(18, 6))
plt.show()

## Summary

The `MKappaStateProfiles` class provides:

1. **Combined Visualization**: M-κ curve with stress-strain profiles in one figure
2. **Interactive Exploration**: Update curvature to see corresponding state
3. **Full StressStrainProfile Features**: Force arrows, value boxes, curvature indicator
4. **Flexible Control**: Set specific κ values or explore at percentage points

### Key Methods

- `MKappaStateProfiles(mkappa)`: Initialize with solved M-κ analysis
- `set_kappa(kappa)`: Update current state
- `plot_mkappa_state()`: Create combined visualization
- `plot_at_percentages([0.25, 0.50, 0.75, 1.0])`: Multiple states at once
- `plot_at_kappa_values([κ1, κ2, ...])`: Specific curvature values

This tool is ideal for:
- Understanding moment-curvature behavior
- Teaching RC cross-section analysis
- Validating design calculations
- Visualizing failure mechanisms